# Threshold Recalibration — no retraining needed

This notebook recalibrates the decision threshold on the **existing** trained model.
The model has AUC ~0.985 but accuracy was capped at ~89.6% because the FPR ≤ 2%
constraint forced threshold = 0.71, causing 18.5% FNR.

Relaxing to FPR ≤ 4% should reach **~93% accuracy** from the same weights.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `tshirt_classifier.keras` when prompted (Cell 1)
3. Authenticate Kaggle to re-download the val/test splits (Cell 2)
4. Run all cells
5. Download the new `threshold.json` from the Files panel

In [ ]:
# ── Cell 1: Upload model ──────────────────────────────────────────────────────
from google.colab import files
import os, json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import roc_auc_score, confusion_matrix

print('Upload tshirt_classifier.keras:')
uploaded = files.upload()

MODEL_PATH = 'tshirt_classifier.keras'
assert os.path.exists(MODEL_PATH), 'Upload failed'
print('Model size:', os.path.getsize(MODEL_PATH) / 1e6, 'MB')

INPUT_SIZE = 300
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# ── Cell 2: Kaggle auth + download val/test splits ────────────────────────────
from google.colab import userdata

token = userdata.get('KAGGLE_TOKEN')
os.environ['KAGGLE_TOKEN'] = token

!pip install -q kaggle
!kaggle datasets download -d rizzvision/rizzvision-tshirt-dataset -p /content/data --unzip

DATASET_DIR = '/content/data'
print('Contents:', os.listdir(DATASET_DIR))

In [ ]:
# ── Cell 3: Load dataset (val + test only) ────────────────────────────────────
def load_dataset(split):
    ds = keras.utils.image_dataset_from_directory(
        os.path.join(DATASET_DIR, split),
        image_size=(INPUT_SIZE, INPUT_SIZE),
        batch_size=None,
        label_mode='binary',
        shuffle=False,
    )
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds  = load_dataset('val')
test_ds = load_dataset('test')
print('Datasets loaded')

In [ ]:
# ── Cell 4: Load model + run val inference ────────────────────────────────────
model = keras.models.load_model(MODEL_PATH, compile=False)
print('Model loaded')

val_probs, val_labels = [], []
for bx, by in val_ds:
    val_probs.extend(model.predict(bx, verbose=0).flatten().tolist())
    val_labels.extend(by.numpy().flatten().tolist())

val_probs  = np.array(val_probs)
val_labels = np.array(val_labels)
print(f'Val AUC: {roc_auc_score(val_labels, val_probs):.4f}')

In [ ]:
# ── Cell 5: Sweep thresholds — show accuracy at each FPR cap ─────────────────
# This helps you understand the accuracy / FPR tradeoff before committing.
neg_count = (val_labels == 0).sum()
pos_count = (val_labels == 1).sum()

print(f'{'FPR cap':>8}  {'Threshold':>10}  {'Accuracy':>10}  {'FPR':>8}  {'FNR':>8}  {'F1':>8}')
print('-' * 60)

for fpr_cap in [0.02, 0.03, 0.04, 0.05, 0.06]:
    best_t, best_f1 = 0.5, 0.0
    for t in np.arange(0.10, 0.95, 0.01):
        preds    = (val_probs >= t).astype(int)
        tp = ((preds == 1) & (val_labels == 1)).sum()
        fp = ((preds == 1) & (val_labels == 0)).sum()
        fn = ((preds == 0) & (val_labels == 1)).sum()
        tn = ((preds == 0) & (val_labels == 0)).sum()
        if fp / neg_count > fpr_cap:
            continue
        prec = tp / (tp + fp + 1e-9)
        rec  = tp / (tp + fn + 1e-9)
        f1   = 2 * prec * rec / (prec + rec + 1e-9)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    if best_t:
        p = (val_probs >= best_t).astype(int)
        tp = ((p==1)&(val_labels==1)).sum(); tn = ((p==0)&(val_labels==0)).sum()
        fp = ((p==1)&(val_labels==0)).sum(); fn = ((p==0)&(val_labels==1)).sum()
        acc = (tp+tn)/len(val_labels)
        fpr_act = fp/neg_count; fnr_act = fn/pos_count
        print(f'{fpr_cap:>8.0%}  {best_t:>10.2f}  {acc:>10.4f}  {fpr_act:>8.4f}  {fnr_act:>8.4f}  {best_f1:>8.4f}')

In [ ]:
# ── Cell 6: Choose FPR cap and save threshold ─────────────────────────────────
# Change FPR_CAP here based on the table above.
# 0.04 (4%) is the recommended sweet spot for ~93% accuracy.
FPR_CAP = 0.04

best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.10, 0.95, 0.01):
    preds    = (val_probs >= t).astype(int)
    tp = ((preds == 1) & (val_labels == 1)).sum()
    fp = ((preds == 1) & (val_labels == 0)).sum()
    fn = ((preds == 0) & (val_labels == 1)).sum()
    if fp / neg_count > FPR_CAP:
        continue
    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    if f1 > best_f1:
        best_f1, best_thresh = f1, float(t)

print(f'Optimal threshold (FPR <= {FPR_CAP:.0%}): {best_thresh:.4f}  (F1: {best_f1:.4f})')
with open('threshold.json', 'w') as f:
    json.dump({'threshold': round(best_thresh, 4)}, f, indent=2)
print('Saved threshold.json')

In [ ]:
# ── Cell 7: Final test set evaluation ────────────────────────────────────────
test_probs, test_labels = [], []
for bx, by in test_ds:
    test_probs.extend(model.predict(bx, verbose=0).flatten().tolist())
    test_labels.extend(by.numpy().flatten().tolist())

test_probs  = np.array(test_probs)
test_labels = np.array(test_labels)
test_preds  = (test_probs >= best_thresh).astype(int)

cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()
accuracy = (tp + tn) / len(test_labels)
fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print('=== Test Set Results ===')
print(f'Accuracy: {accuracy:.4f}  (target >= 0.92)')
print(f'AUC-ROC:  {roc_auc_score(test_labels, test_probs):.4f}')
print(f'FPR:      {fpr:.4f}  (target <= {FPR_CAP:.0%})')
print(f'FNR:      {fnr:.4f}')
print(f'Threshold used: {best_thresh}')
print('\nConfusion matrix:')
print(cm)

if accuracy >= 0.92 and fpr <= FPR_CAP:
    print('\n✓ Target met — download threshold.json and commit to backend/model/')
else:
    print('\n✗ Target not met — may need full retrain with more data/epochs')

In [ ]:
# ── Cell 8: Download threshold.json ──────────────────────────────────────────
files.download('threshold.json')